In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#バージョン確認　基本的に不要
#!nvidia-smi

In [ ]:
#パッケージのインストール 音声に発話していない空白時間を除去するために、VAD(Voice Activity Detection)音声区間検出を使います
!pip install -q torchaudio

SAMPLING_RATE = 16000

import torch
torch.set_num_threads(1)

from IPython.display import Audio
from pprint import pprint

In [ ]:
USE_ONNX = False # change this to True if you want to test onnx model
if USE_ONNX:
    !pip install -q onnxruntime

model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                              model='silero_vad',
                              force_reload=True,
                              onnx=USE_ONNX)

(get_speech_timestamps,
 save_audio,
 read_audio,
 VADIterator,
 collect_chunks) = utils

/usr/local/lib/python3.10/dist-packages/torch/hub.py:294: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip


In [ ]:
import os
directory = '/content/drive/MyDrive/ja_voice_finetuning/clips/'
out_directory = '/content/drive/MyDrive/ja_voice_finetuning/silero-clips/'

# 指定されたディレクトリ内の全てのファイルをループ処理
for filename in os.listdir(directory):
    if filename.endswith(".mp3"):  # ファイルが.mp3で終わる場合
        file_path = os.path.join(directory, filename)  # ファイルのフルパスを取得
        print(f'Processing file: {file_path}')

        # read_audio とは、ファイルを読み込んで適切な形式（例: numpy array）に変換する関数です
        audio = read_audio(file_path, sampling_rate=SAMPLING_RATE)

        # get_speech_timestamps は、音声のタイムスタンプを検出する関数です
        speech_timestamps = get_speech_timestamps(audio, model, sampling_rate=SAMPLING_RATE)
        print(speech_timestamps)

        if speech_timestamps:  # タイムスタンプリストが空でない場合
            # 処理された音声を保存する関数を呼び出す
            output_path = os.path.join(out_directory, filename)
            print(output_path)
            save_audio(output_path, collect_chunks(speech_timestamps, audio), sampling_rate=SAMPLING_RATE)
        else:
            print(f"No speech detected in file: {file_path}")




Processing file: /content/drive/MyDrive/ja_voice_finetuning/clips/record_audio_2024-5-4 14-47-40.mp3
[{'start': 17440, 'end': 58336}, {'start': 59936, 'end': 73696}]
/content/drive/MyDrive/ja_voice_finetuning/silero-clips/record_audio_2024-5-4 14-47-40.mp3
Processing file: /content/drive/MyDrive/ja_voice_finetuning/clips/record_audio_2024-5-3 23-45-17.mp3
[{'start': 43040, 'end': 52704}, {'start': 58400, 'end': 71136}, {'start': 75296, 'end': 104928}]
/content/drive/MyDrive/ja_voice_finetuning/silero-clips/record_audio_2024-5-3 23-45-17.mp3
Processing file: /content/drive/MyDrive/ja_voice_finetuning/clips/record_audio_2024-5-4 14-47-26.mp3
[{'start': 50720, 'end': 79840}, {'start': 83488, 'end': 116160}]
/content/drive/MyDrive/ja_voice_finetuning/silero-clips/record_audio_2024-5-4 14-47-26.mp3


In [ ]:
out_directory

'/content/drive/MyDrive/ja_voice_finetuning/silero-clips/'

In [ ]:
#正解ラベルの作成
!pip install pydub openai-whisper
!pip install kora

#GoogleDriveへのリンクを取得する関数を定義
from kora.xattr import get_id
def get_gdrive_link(file_path):
  fid = get_id(file_path)
  print(f"https://drive.google.com/file/d/{format(fid)}/view")
  return f"https://drive.google.com/file/d/{format(fid)}/view"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.6/798.6 kB 14.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.9 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20231117-py3-none-any.whl size=801358 sha256=1bfe6683463c11fa50e198b9002cf5372f26f48ae625566a64e9714a3520072c
  Stored in directory: /root/.cache/pip/wheels/d0/85/e1/9361b4cbea7dd4b7f6702fa4c3afc94877952eeb2b62f45f56
Successfully built openai-whisper
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.5 MB/s eta 0:00:00


In [ ]:
#音声ファイルから文字起こしした内容を、CSVに書き出し

import whisper
import torch
import os
import csv  # CSVファイル出力のために必要

# 現在利用可能なデバイスを確認
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Whisperモデルをロード
model = whisper.load_model("large-v3", device=device)

# 出力先のCSVファイルパスを定義
output_csv_file = '/content/drive/MyDrive/ja_voice_finetuning/train.csv'


# CSVファイルを開き、結果を書き込む
with open(output_csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # CSVのヘッダーを書き込む
    writer.writerow(['url', 'path','sampling_rate', 'correct', 'whisper'])

    # 指定されたディレクトリ内の全てのファイルをループ処理
    for filename in os.listdir(out_directory):
        if filename.endswith(".mp3"):  # ファイルが.mp3で終わる場合
            file_path = os.path.join(out_directory, filename)  # ファイルのフルパスを取得
            print(f'Processing file: {file_path}')
            result = model.transcribe(file_path, language="ja")
            # 結果をCSVに書き込む（ここではurlとcontentは空とする）
            # https://drive.google.com/file/d/1ISt6Pg2eJFuHvV1Fkm2NZBvzAJ4grewq/view?usp=drive_link
            writer.writerow([get_gdrive_link(file_path), file_path, SAMPLING_RATE ,'', result["text"]])
            print(result["text"])

print(f"Processing complete. Results saved to {output_csv_file}")

Using device: cuda


100%|█████████████████████████████████████| 2.88G/2.88G [02:34<00:00, 20.0MiB/s]


Processing file: /content/drive/MyDrive/ja_voice_finetuning/silero-clips/record_audio_2024-5-4 14-47-40.mp3
https://drive.google.com/file/d/10AGag0EOllNMy8aT6iqiTz7bT8zoKyI6/view
ヤマハ UX3-2254-625
Processing file: /content/drive/MyDrive/ja_voice_finetuning/silero-clips/record_audio_2024-5-3 23-45-17.mp3
https://drive.google.com/file/d/104756seHJ7nFxpDIIGFBJt2uRDuZwb-p/view
ヤナハ13駅 2256254
Processing file: /content/drive/MyDrive/ja_voice_finetuning/silero-clips/record_audio_2024-5-4 14-47-26.mp3
https://drive.google.com/file/d/104310VfPcA6xo-cnEGw8nArImAwGR4uQ/view
ヤマハU2H5525343
Processing complete. Results saved to /content/drive/MyDrive/ja_voice_finetuning/train.csv
